In [1]:
from steer_core.DataManager import DataManager
import steer_opencell_design as ocd

In [2]:
# User Inputs

####################

table_name = 'cell_references' #### change this to cell_teardowns for teardowns
chemistry = 'Li/Li+'   #### change to 'Li/Li+' for lithium-ion cells
cell_name = "NMC811 Stacked Pouch Cell"

#####################

In [3]:
# set some standard materials

conductive_additive = ocd.ConductiveAdditive.from_database("Super P")
binder = ocd.Binder.from_database("PVDF")
insulation = ocd.InsulationMaterial.from_database("Aluminium Oxide, 95%")
separator_material = ocd.SeparatorMaterial.from_database('Polyethylene')
tape_material = ocd.TapeMaterial.from_database("Kapton")
prismatic_material = ocd.PrismaticContainerMaterial.from_database("Steel")

In [4]:
# Create the cathode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database('Aluminum')

cathode_current_collector=ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=300,
    height=280,
    tab_height=30,
    tab_position=70,
    tab_width=80,
    thickness=10,
    insulation_width=2
)

cathode_active_material = ocd.CathodeMaterial.from_database("NMC811")

cathode_formulation = ocd.CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = ocd.Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3.1,
    mass_loading=14,
    insulation_material=insulation,
    insulation_thickness=3
)


In [5]:
# Create the anode

cathode_current_collector_material = ocd.CurrentCollectorMaterial.from_database("Copper")

anode_current_collector = ocd.PunchedCurrentCollector(
    material=cathode_current_collector_material,
    width=300,
    height=280,
    tab_height=30,
    tab_position=230,
    tab_width=80,
    thickness=10
)

anode_active_material = ocd.AnodeMaterial.from_database("Synthetic Graphite")

anode_formulation = ocd.AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = ocd.Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=9
)

In [6]:
# create the layup 

separator = ocd.Separator(
    material=separator_material, 
    thickness=12,
    width=280,
    length=300
)

my_layup = ocd.ZFoldMonoLayer(
    cathode=my_cathode,
    anode=my_anode,
    separator=separator,
)

my_layup.get_top_down_view().show()


In [7]:
# create the stack assembly

my_stack = ocd.ZFoldStack(
    layup=my_layup,
    n_layers=40,
    additional_separator_wraps=3
)

# looks best in safari
my_stack.get_side_view().show()

In [8]:
# make the electrolyte

my_electrolyte = ocd.Electrolyte(
    name="1M NaPF6 in EC:PC:DMC (1:1:1 wt%)",
    density=1.2,
    specific_cost=10,
    color="#FF9D00"
)


In [9]:
# make the encapsulation

top_laminate = ocd.LaminateSheet(
    areal_cost=0.06,
    density=1.4,
    thickness=80
)

bottom_laminate = ocd.LaminateSheet(
    areal_cost=0.06,
    density=1.4,
    thickness=80
)

cathode_terminal_connector = ocd.PouchTerminal(
    material=prismatic_material,
    width=50,
    length=10,
    thickness=1
)

anode_terminal_connector = ocd.PouchTerminal(
    material=prismatic_material,
    width=50,
    length=10,
    thickness=1
)

encapsulation = ocd.PouchEncapsulation(
    top_laminate=top_laminate,
    bottom_laminate=bottom_laminate,
    cathode_terminal=cathode_terminal_connector,
    anode_terminal=anode_terminal_connector
)


In [10]:
# make the cell

cell = ocd.PouchCell(
    reference_electrode_assembly=my_stack,
    electrolyte=my_electrolyte,
    electrolyte_overfill=0.1,
    encapsulation=encapsulation,
    n_electrode_assembly=1,
    clipped_tab_length=10,
    name=cell_name
)

# looks better in safari
cell.get_side_view().show()
cell.get_top_down_view().show()

In [11]:
cell.plot_mass_breakdown(width=800, height=800)

In [12]:
cell.plot_cost_breakdown(width=800, height=800)

In [13]:
cell.get_capacity_plot(width=1300, height=800).show()

In [14]:
print(f"Cost ($): {cell.cost}")
print(f"Mass (g): {cell.mass}")
print(f"Energy Density (Wh/L): {cell.volumetric_energy}")
print(f"Energy (Wh): {cell.energy}")
print(f"Energy Density (Wh/kg): {cell.specific_energy}")
print(f"Normalized Cost ($/kWh): {cell.cost_per_energy}")

Cost ($): 41.01
Mass (g): 2828.35
Energy Density (Wh/L): 564.3
Energy (Wh): 658.82
Energy Density (Wh/kg): 232.93
Normalized Cost ($/kWh): 62.24


In [ ]:
# import pandas as pd
# import datetime as dt
# from steer_opencell_design import __version__


# db = DataManager()

# db.remove_data(table_name=table_name, condition=f"name = '{cell.name}'")

# internal_construction = cell.reference_electrode_assembly.__class__.__name__
# form_factor = cell.__class__.__name__

# # insert the cell into the database
# db.insert_data(table_name=table_name, data=pd.DataFrame({
#     'name': [cell.name],
#     'object': [cell.serialize()],
#     'form_factor': [form_factor],
#     'internal_construction': [internal_construction],
#     'date_created': [dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")],
#     'version': [__version__],
#     'chemistry': [chemistry]
# }))

# db.get_data(table_name)

,name,object,form_factor,internal_construction,date_created,version,chemistry
0,LFP Cylindrical Tabless Cell,gASVLVgAAAAAAACMOXN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,CylindricalCell,WoundJellyRoll,2025-12-11 17:02:17,0.4.12,Li/Li+
1,LFP Flat Wound Prismatic Cell,gASVZloAAAAAAACMN3N0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,PrismaticCell,FlatWoundJellyRoll,2025-12-11 17:02:54,0.4.12,Li/Li+
2,NFM111 Cylindrical Tabless Cell,gASV7Q0AAAAAAACMOXN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,CylindricalCell,WoundJellyRoll,2025-12-11 17:03:01,0.4.12,Na/Na+
3,NFM111 Z-Fold Prismatic Cell,gASVbwcAAAAAAACMN3N0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,PrismaticCell,ZFoldStack,2025-12-11 17:03:08,0.4.12,Na/Na+
4,NFPP Flat Wound Prismatic Cell,gASVvhABAAAAAACMN3N0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,PrismaticCell,FlatWoundJellyRoll,2025-12-11 17:03:17,0.4.12,Na/Na+
5,NMC811 Stacked Pouch Cell,gASVzQgAAAAAAACMM3N0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,PouchCell,ZFoldStack,2025-12-11 17:03:24,0.4.12,Li/Li+
